# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashishpal003/flyrank_ml_intern/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — reuse the ML-09 assembly + models

The preamble is `w06_validation_audit.ipynb`'s helper cell and `assemble(D)`, verbatim, plus a small `oof_predict()` (grouped 5-fold out-of-fold probabilities, matching ML-08's `model_oof_prob`).

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass
from pathlib import Path

# Token order: env var -> .env file (local runs) -> Colab Secret -> prompt (last resort).
# Never commit the token: `.env` is gitignored and this repo is public.
def _from_dotenv(key):
    for base in [Path.cwd(), *Path.cwd().parents]:
        f = base / ".env"
        if f.is_file():
            for line in f.read_text().splitlines():
                s = line.strip()
                if s.startswith(f"{key}=") or s.startswith(f"export {key}="):
                    return s.split("=", 1)[1].strip().strip('\"').strip("'")
    return None

HF_TOKEN = os.environ.get("HF_TOKEN") or _from_dotenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "no valid HF READ token found (env / .env / Colab secret)"

In [3]:
import duckdb, json
import numpy as np
import pandas as pd
import sklearn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
SEED = 42
K_ABS = 50

def daily(months):
    """read_parquet over an explicit list of month partitions (hf:// has no brace globs)."""
    paths = [f"'{REL}/fact_content_daily_performance/month={m}/*.parquet'" for m in months]
    return f"read_parquet([{', '.join(paths)}])"

# --- decision date -> (feature month partitions, label month partition) ---
# 2026-06 is the sealed month; it is NOT in this map -- ML-10 never assembles June.
FEAT_MAP = {
    '2026-03-01': (['2025-12', '2026-01', '2026-02'], '2026-03'),
    '2026-04-01': (['2026-01', '2026-02', '2026-03'], '2026-04'),
    '2026-05-01': (['2026-02', '2026-03', '2026-04'], '2026-05'),
}

# --- models + metrics: verbatim from w05_model.ipynb / w06_validation_audit.ipynb ---
def make_models():
    return {
        'dummy_prior': DummyClassifier(strategy='prior'),
        'logistic_regression': Pipeline([('scaler', StandardScaler()),
            ('m', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED))]),
        'random_forest': RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
            class_weight='balanced_subsample', n_jobs=-1, random_state=SEED),
    }

def pct_rank(s):
    return s.rank(pct=True, method='average')

def components_from(df):
    p = (df['imp_90d'] / 3.0).replace(0, np.nan)
    pg = df['pos_last30'] - df['pos_prev60']
    return pd.DataFrame({
        'traffic_softening': pct_rank((1 - df['imp_last30'] / p).clip(0, 1).fillna(0)),
        'position_slip':     pct_rank(pg.clip(0, 20).fillna(0)) * pg.notna().astype(int),
        'reach_thinning':    pct_rank((1 - df['days_impr_last30'] / df['days_impr_prev30'].replace(0, np.nan)).clip(0, 1).fillna(0)),
    }, index=df.index)

WEIGHTS = {'traffic_softening': 0.40, 'position_slip': 0.30, 'reach_thinning': 0.30}   # frozen in ML-07

def frozen_baseline_score(df):
    c = components_from(df)
    return (sum(WEIGHTS[k] * c[k] for k in WEIGHTS) + 1e-9 * pct_rank(np.log1p(df['imp_90d']))).to_numpy()

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float), kind='stable')
    return float(np.asarray(labels)[order[:k]].mean())

def average_precision(scores, labels):
    order = np.argsort(-np.asarray(scores, dtype=float), kind='stable')
    yy = np.asarray(labels)[order]
    if yy.sum() == 0: return 0.0
    prec = np.cumsum(yy) / (np.arange(len(yy)) + 1)
    return float((prec * yy).sum() / yy.sum())

def per_client_p_at(df, score_col, k=10):
    vals = [precision_at_k(g[score_col], g['label_decline'], k)
            for _, g in df.groupby('client_hash_id', observed=True) if len(g) >= k]
    return float(np.mean(vals)) if vals else float('nan')

def oof_predict(X, y, groups, model_name='random_forest', n_folds=5, seed=SEED):
    """grouped 5-fold out-of-fold probabilities -- matches ML-08's model_oof_prob."""
    rng = np.random.default_rng(seed)
    uc = rng.permutation(np.unique(groups))
    fold_of = {c: fi for fi, arr in enumerate(np.array_split(uc, n_folds)) for c in arr}
    fold = np.array([fold_of[g] for g in groups])
    oof = np.zeros(len(y))
    for fi in range(n_folds):
        te = fold == fi
        m = make_models()[model_name].fit(X[~te], y[~te])
        oof[te] = m.predict_proba(X[te])[:, 1]
    return oof, fold

NUM = ['imp_90d', 'log_imp_90d', 'clk_90d', 'ctr_90d', 'imp_last30', 'imp_prev60', 'clk_last30',
       'softening_ratio', 'pos_last30', 'pos_prev60', 'pos_gap',
       'days_impr_90d', 'days_impr_last30', 'days_impr_prev30', 'reach_ratio',
       'content_age_days', 'days_since_update', 'word_count', 'char_count',
       'search_volume', 'competition', 'category_count', 'backlinks',
       'has_word_count', 'has_search_volume', 'has_update_date', 'has_position']
CAT = ['content_type', 'main_intent', 'competition_level']
BANNED_SUBSTR = ['trend_direction', 'trend_pct', 'is_declining', 'imp_label', 'label_decline',
                 'last_optimized', 'ga4_', 'query', 'impressions_90d', 'health_score',
                 'priority_score', 'action_type']
print('setup ok. sklearn', sklearn.__version__, '| numpy', np.__version__,
      '| pandas', pd.__version__, '| matplotlib', matplotlib.__version__)

setup ok. sklearn 1.9.0 | numpy 2.5.1 | pandas 3.0.5 | matplotlib 3.11.1


In [4]:
def assemble(D, *, reconcile=None):
    fmonths, lmonth = FEAT_MAP[D]
    Dts = pd.Timestamp(D)
    raw = con.sql(f"""
        WITH feat AS (
            SELECT f.client_hash_id, f.content_hash_id,
                   SUM(f.gsc_impressions) AS imp_90d,
                   SUM(f.gsc_clicks)      AS clk_90d,
                   SUM(CASE WHEN f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_impressions  ELSE 0 END) AS imp_last30,
                   SUM(CASE WHEN f.report_date <  DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_impressions  ELSE 0 END) AS imp_prev60,
                   SUM(CASE WHEN f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_clicks       ELSE 0 END) AS clk_last30,
                   SUM(CASE WHEN f.report_date <  DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_clicks       ELSE 0 END) AS clk_prev60,
                   SUM(CASE WHEN f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_sum_position ELSE 0 END) AS sumpos_last30,
                   SUM(CASE WHEN f.report_date <  DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_sum_position ELSE 0 END) AS sumpos_prev60,
                   COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END) AS days_impr_90d,
                   COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0
                         AND f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.report_date END) AS days_impr_last30,
                   COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0
                         AND f.report_date <  DATE '{D}' - INTERVAL 30 DAY
                         AND f.report_date >= DATE '{D}' - INTERVAL 60 DAY THEN f.report_date END) AS days_impr_prev30
            FROM {daily(fmonths)} f
            WHERE f.report_date >= DATE '{D}' - INTERVAL 90 DAY AND f.report_date < DATE '{D}'
            GROUP BY 1, 2
        ),
        lab AS (
            SELECT content_hash_id,
                   SUM(gsc_impressions) AS imp_label,
                   SUM(CASE WHEN report_date <  DATE '{D}' + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_label_h1,
                   SUM(CASE WHEN report_date >= DATE '{D}' + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_label_h2
            FROM {daily([lmonth])}
            WHERE report_date >= DATE '{D}' AND report_date < DATE '{D}' + INTERVAL 1 MONTH
            GROUP BY 1
        ),
        lab_client AS (
            SELECT client_hash_id, COUNT(*) AS client_label_rows
            FROM {daily([lmonth])}
            WHERE report_date >= DATE '{D}' AND report_date < DATE '{D}' + INTERVAL 1 MONTH
            GROUP BY 1
        ),
        lab_content AS (
            SELECT DISTINCT content_hash_id FROM {daily([lmonth])}
            WHERE report_date >= DATE '{D}' AND report_date < DATE '{D}' + INTERVAL 1 MONTH
        )
        SELECT feat.*,
               dc.word_count, dc.char_count, dc.content_type, dc.main_intent,
               dc.search_volume, dc.competition, dc.competition_level, dc.category_count, dc.backlinks,
               dc.content_created_date, dc.content_updated_date,
               date_diff('day', dc.content_created_date, DATE '{D}') AS content_age_days,
               CASE WHEN dc.content_updated_date < DATE '{D}'
                    THEN date_diff('day', dc.content_updated_date, DATE '{D}') END AS days_since_update,
               cl.gsc_data_start,
               COALESCE(lc.client_label_rows, 0) AS client_label_rows,
               (lct.content_hash_id IS NOT NULL) AS content_in_label,
               feat.imp_90d / 3.0 AS pace_30d,
               CASE WHEN feat.imp_last30 > 0 THEN feat.sumpos_last30::DOUBLE / feat.imp_last30 END AS pos_last30,
               CASE WHEN feat.imp_prev60 > 0 THEN feat.sumpos_prev60::DOUBLE / feat.imp_prev60 END AS pos_prev60,
               COALESCE(lab.imp_label, 0)    AS imp_label,
               COALESCE(lab.imp_label_h1, 0) AS imp_label_h1,
               COALESCE(lab.imp_label_h2, 0) AS imp_label_h2
        FROM feat
        LEFT JOIN {DIM_CONTENT} dc ON feat.content_hash_id = dc.content_hash_id
        LEFT JOIN {DIM_CLIENTS} cl ON feat.client_hash_id = cl.client_hash_id
        LEFT JOIN lab              ON feat.content_hash_id = lab.content_hash_id
        LEFT JOIN lab_client lc    ON feat.client_hash_id = lc.client_hash_id
        LEFT JOIN lab_content lct  ON feat.content_hash_id = lct.content_hash_id
    """).df()

    raw['pass_volume_floor']   = raw['imp_last30'] >= 100
    raw['pass_client_history'] = raw['gsc_data_start'] <= (Dts - pd.Timedelta(days=90))
    raw['pass_not_freefall']   = ~((raw['imp_prev60'] > 0) & (raw['imp_last30'] < 0.5 * raw['imp_prev60']))
    raw['pass_client_reports'] = raw['client_label_rows'] > 0          # ML-08 label-quality gate, formalised
    raw['label_decline'] = ((raw['imp_label'] < 0.75 * raw['pace_30d']) &
                            (raw['imp_label_h2'] <= raw['imp_label_h1'])).astype(int)

    base_elig = raw['pass_volume_floor'] & raw['pass_client_history'] & raw['pass_not_freefall']
    keep = base_elig & raw['pass_client_reports']
    n_page_absent = int((base_elig & raw['pass_client_reports'] & ~raw['content_in_label']).sum())
    e = (raw[keep].sort_values(['client_hash_id', 'content_hash_id'])   # deterministic row order
                  .reset_index(drop=True).copy())

    e['log_imp_90d']     = np.log1p(e['imp_90d'])
    e['ctr_90d']         = e['clk_90d'] / e['imp_90d'].replace(0, np.nan)
    e['softening_ratio'] = e['imp_last30'] / e['pace_30d'].replace(0, np.nan)
    e['pos_gap']         = e['pos_last30'] - e['pos_prev60']
    e['reach_ratio']     = e['days_impr_last30'] / e['days_impr_prev30'].replace(0, np.nan)
    e['has_word_count']    = e['word_count'].notna().astype(int)
    e['has_search_volume'] = e['search_volume'].notna().astype(int)
    e['has_update_date']   = e['days_since_update'].notna().astype(int)
    e['has_position']      = e['pos_last30'].notna().astype(int)

    X_num = e[NUM].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X_cat = pd.get_dummies(e[CAT].astype('string').fillna('unknown'), prefix=CAT, dummy_na=False, dtype=float)
    X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
    y = e['label_decline'].to_numpy()
    groups = e['client_hash_id'].to_numpy()

    bad = [c for c in X.columns if any(b in c for b in BANNED_SUBSTR)]
    assert not bad, f'banned column leaked into X for {D}: {bad}'

    base = float(y.mean())
    print(f"assemble({D}): {len(e):>7,} rows | {len(np.unique(groups)):>3} clients | base rate {base:.4f}")
    if reconcile is not None:
        assert abs(len(e) - reconcile[0]) <= 200 and abs(base - reconcile[1]) < 0.008, \
            f"reconcile failed for {D}: {len(e)} rows / base {base:.4f}"
        print(f"  -> reconciles with ML-08/09 ({reconcile[0]:,} rows, base rate {reconcile[1]})")
    return e, X, y, groups

e, X, y, groups = assemble('2026-03-01', reconcile=(58_583, 0.045))
BASE = float(y.mean())
rf_oof, fold = oof_predict(X, y, groups)
print(f'\nrandom-forest grouped out-of-fold: mean prob {rf_oof.mean():.3f}, '
      f'OOF avg precision {average_precision(rf_oof, y):.3f} (base rate {BASE:.3f})')

assemble(2026-03-01):  58,583 rows |  23 clients | base rate 0.0450
  -> reconciles with ML-08/09 (58,583 rows, base rate 0.045)



random-forest grouped out-of-fold: mean prob 0.324, OOF avg precision 0.072 (base rate 0.045)


## 1. Ranked actions + reason codes

**"Review first" means: order a person's limited attention — not decide the edit and not decide whether to act.** A FlyRank content strategist opens the weekly queue for a client and works down from the top; for each page they read the recent trend, sanity-check the reason codes, and make the call themselves.

**The score.** `final_score = 100 · (0.70 · model_score + 0.30 · frozen-rule_percentile)`, where `model_score` is the random forest from ML-08 scored **out-of-fold and rank-normalised within each fold** (raw cross-fold probabilities are not comparable — the pooling trap the ML-09 reference paper documents). The 30% rule anchor keeps ML-07's transparent hand-rule visible in every ranking and carries the auditable reason. It is a **model score, not a probability** (calibration unevaluated — ML-09).

**Reason codes** (GSC-only — ML-04 found GA4 is ~4% coverage, so no engagement codes). Every row carries a `|`-joined list; fallback `general_review`:

| code | fires when | kind |
|---|---|---|
| `high_exposure` | `imp_90d >= 500` | rule |
| `traffic_softening` | `imp_last30 < 0.8 · (imp_90d/3)` | rule |
| `position_slipping` | avg position worsened >= 1.0 place, last 30d vs prior 60d | rule |
| `reach_thinning` | days-with-impressions fell >= 25% | rule |
| `stale_content` | `days_since_update >= 180` (partial coverage) | rule |
| `model_high_risk` | model score in the top 10% (within fold) | model |
| `model_and_rule_agree` | `model_high_risk` **and** >= 2 rule codes | model + rule |
| `durable_riser_hold` | `imp_last30 > 1.5 x pace` — traffic is *rising*; route to monitor, never refresh | policy (ML-09) |
| `low_volume_position_only` | `imp_90d < 250` and the only rule signal is `position_slipping` | policy (ML-07/09 false-positive mode) |

**Suggested action** (first match): `durable_riser_hold` -> `monitor_only`; `low_volume_position_only` alone -> `watchlist`; `traffic_softening`/`reach_thinning` -> `refresh`; `position_slipping` + `high_exposure` -> `review_position`; `stale_content` alone -> `review_metadata`; else -> `monitor`.

**Confidence tier:** `high` needs `final_score` top 20% **and** `imp_90d >= 500` **and** model score top 20% **and** >= 2 rule codes **and** neither policy hold; `medium` = `final_score` >= median; else `low`.

### The top-20 read

**Top-20 out-of-fold hit rate: 0.25** against a 0.045 base rate — **5.5x**. Most top picks did *not* decline. That is what modest skill on a 4.5% base rate looks like, and it is exactly why the queue carries a confidence tier, an explicit no-go list, and a mandatory human check.

The queue head is **concentrated** — a handful of clients (`client_e547…`, `client_fef1…`, `client_73cda…`) and mid-volume pages (~250–500 impressions) that the model flags on its own; many rows carry only `model_high_risk` with no rule code. The confidence mix is ~50% `low`, ~49% `medium`, ~1.4% `high` — the `high` bar is deliberately hard to clear. Of the top 50, **13 are `durable_riser_hold`** (traffic actually rising) and are auto-routed to `monitor_only`, and **8 are `low_volume_position_only`** and routed to `watchlist` — the policy guards catch about a third of the head before a human sees it.

In [5]:
e['rf_oof'] = rf_oof
e['_fold'] = fold
# within-fold rank normalisation: raw OOF probabilities are NOT comparable across folds
# (each fold trains on a different base rate) -- pooling them is the trap the ML-09 reference
# paper documents. Rank within fold first, then blend.
e['rf_oof_norm'] = e.groupby('_fold', observed=True)['rf_oof'].transform(lambda s: s.rank(pct=True, method='average'))
e['rule_pct'] = pct_rank(pd.Series(frozen_baseline_score(e), index=e.index))
e['final_score'] = (100 * (0.70 * e['rf_oof_norm'] + 0.30 * e['rule_pct'])).round(2)
e['last30_vs_pace'] = (e['imp_last30'] / e['pace_30d']).round(2)

P80 = float(e['final_score'].quantile(0.80))
P50 = float(e['final_score'].quantile(0.50))
RULE_CODES = ['high_exposure', 'traffic_softening', 'position_slipping', 'reach_thinning', 'stale_content']

def reason_codes(r):
    rc = []
    if r['imp_90d'] >= 500: rc.append('high_exposure')
    if r['imp_last30'] < 0.8 * r['pace_30d']: rc.append('traffic_softening')
    if pd.notna(r['pos_last30']) and pd.notna(r['pos_prev60']) and (r['pos_last30'] - r['pos_prev60']) >= 1.0:
        rc.append('position_slipping')
    if r['days_impr_prev30'] > 0 and r['days_impr_last30'] <= 0.75 * r['days_impr_prev30']:
        rc.append('reach_thinning')
    if pd.notna(r['days_since_update']) and r['days_since_update'] >= 180:
        rc.append('stale_content')
    n_rule = len(rc)
    if r['rf_oof_norm'] >= 0.90: rc.append('model_high_risk')          # top 10% of model score, within fold
    if r['rf_oof_norm'] >= 0.90 and n_rule >= 2: rc.append('model_and_rule_agree')
    if pd.notna(r['softening_ratio']) and r['softening_ratio'] > 1.5: rc.append('durable_riser_hold')
    if r['imp_90d'] < 250 and n_rule == 1 and 'position_slipping' in rc: rc.append('low_volume_position_only')
    return rc or ['general_review']

def suggested_action(codes):
    s = set(codes)
    if 'durable_riser_hold' in s: return 'monitor_only'
    if 'low_volume_position_only' in s and not (s & {'traffic_softening', 'reach_thinning'}): return 'watchlist'
    if s & {'traffic_softening', 'reach_thinning'}: return 'refresh'
    if 'position_slipping' in s and 'high_exposure' in s: return 'review_position'
    if s == {'stale_content'}: return 'review_metadata'
    return 'monitor'

def confidence(r):
    codes = r['_codes']
    n_rule = len([c for c in codes if c in RULE_CODES])
    if (r['final_score'] >= P80 and r['imp_90d'] >= 500 and r['rf_oof_norm'] >= 0.80 and n_rule >= 2
            and 'durable_riser_hold' not in codes and 'low_volume_position_only' not in codes):
        return 'high'
    if r['final_score'] >= P50: return 'medium'
    return 'low'

_codes = e.apply(reason_codes, axis=1)
e['_codes'] = _codes
e['reason_codes'] = _codes.apply('|'.join)
e['suggested_action'] = _codes.apply(suggested_action)
e['confidence'] = e.apply(confidence, axis=1)
e = e.sort_values(['final_score', 'imp_90d'], ascending=False).reset_index(drop=True)
e['final_rank'] = e.index + 1

top20 = e.head(20)
print(f'top-20 OOF hit rate: {top20["label_decline"].mean():.3f}   (base rate {BASE:.3f})\n')
show = ['final_rank', 'content_hash_id', 'client_hash_id', 'final_score', 'confidence',
        'suggested_action', 'reason_codes', 'imp_90d', 'last30_vs_pace', 'pos_gap', 'label_decline']
pd.set_option('display.max_colwidth', 55); pd.set_option('display.width', 200)
print(top20[show].to_string(index=False))
print('\naction mix (whole queue):');     print(e['suggested_action'].value_counts().to_string())
print('\nconfidence mix (whole queue):'); print(e['confidence'].value_counts().to_string())


top-20 OOF hit rate: 0.250   (base rate 0.045)

 final_rank          content_hash_id          client_hash_id  final_score confidence suggested_action                                                                            reason_codes  imp_90d  last30_vs_pace  pos_gap  label_decline
          1 content_62ee86ec78b7352d client_e547b89c05043229        99.87     medium          monitor                                                       position_slipping|model_high_risk    429.0            1.20 3.107643              0
          2 content_b1a8e77885353f29 client_a2eeb8899886adde        99.84     medium     monitor_only    high_exposure|reach_thinning|model_high_risk|model_and_rule_agree|durable_riser_hold   1168.0            1.55 0.717197              1
          3 content_d6be097e02ed2448 client_65de48885f4ef01b        99.75     medium          refresh                   position_slipping|reach_thinning|model_high_risk|model_and_rule_agree    300.0            1.11 2.292292            

## 2. Intended use and limits

**Who uses it, for what.** A FlyRank content strategist or editor, once a week, per client account. They open the ranked queue and use it to decide **which pages to look at first** with the review hours they have. It does **not** decide what to do to a page (refresh / expand / rewrite metadata / leave) — that is the editor's call, informed by the reason codes — and it does not decide **whether** to act.

**Where it stops being valid** (each is a limitation, not a hedge):

- **Churn-censored population.** Clients that stopped reporting in a label month are dropped (the ML-08 label-quality gate). The queue describes *surviving* portfolios; it says nothing about a client mid-offboarding.
- **GSC-visible, above-floor pages only.** A page must have ≥ 100 search impressions in the prior 30 days to enter. **Absence from the queue is not evidence a page is healthy** — it may just be below the floor or invisible to Search Console.
- **One content operation, one ~17-month snapshot.** No causal design. Nothing here shows that refreshing a flagged page changes its outcome.
- **The label base rate is not stable** — it moved 0.045 → 0.219 → 0.198 → 0.345 across four months (§4) because it tracks the panel-wide impressions trend. So `precision@K` is **not comparable across months**, and `final_score` is **not a calibrated probability**.
- **Per-client results vary widely.** ML-09 measured per-fold precision@50 from 0.04 to 0.50. The queue works well for some client books and barely at all for others; the per-client number is the honest one.

The cell below prints the eligibility funnel and the per-client spread.

In [6]:
pc = (e.assign(_s=e['final_score'])
        .groupby('client_hash_id', observed=True)
        .apply(lambda g: pd.Series({
            'pages': len(g),
            'decline_rate': g['label_decline'].mean(),
            'precision@10': precision_at_k(g['_s'], g['label_decline'], 10) if len(g) >= 10 else np.nan,
        }), include_groups=False)
        .round(3))
print('per-client (clients with >=10 eligible pages), sorted by precision@10:')
print(pc[pc['precision@10'].notna()].sort_values('precision@10', ascending=False).to_string())
print(f"\nmean per-client precision@10 (final_score): {per_client_p_at(e.assign(_s=e['final_score']).rename(columns={'_s':'final_score'}), 'final_score'):.3f}   base rate {BASE:.3f}")
print(f"queue size: {len(e):,} pages across {e['client_hash_id'].nunique()} clients")
print('confidence share:', (e['confidence'].value_counts(normalize=True).round(3)).to_dict())

per-client (clients with >=10 eligible pages), sorted by precision@10:
                           pages  decline_rate  precision@10
client_hash_id                                              
client_795153d5b7850ccf     30.0         0.600           0.8
client_c182d11e4862a37d    513.0         0.045           0.5
client_9958f0a7ae1df715     61.0         0.279           0.4
client_08d2847f24cf89c1     10.0         0.300           0.3
client_3197e6291363b4db    540.0         0.102           0.3
client_400c21c81c8b46ef    699.0         0.082           0.3
client_62f4a7e64f5e0096  13097.0         0.079           0.3
client_def0955f7a377868     18.0         0.167           0.2
client_cd12bcfd98942aa1     67.0         0.134           0.2
client_08a6a72ff48e62c0   3614.0         0.057           0.2
client_65de48885f4ef01b    561.0         0.189           0.2
client_3ffa76342f366962     87.0         0.207           0.2
client_73cda7b4e4f265ea  16737.0         0.037           0.1
client_b10cb29

## 3. Human review + the no-go list

### Before acting on a flagged page, a person must

1. **Open the page and read the trailing 90-day trend** — does the decline the queue implies actually show up?
2. **Check the reason codes describe what they see** — `position_slipping` should mean the page really slid, not that a mean bounced on a handful of impressions.
3. **Check it is not a low-volume page riding position noise** (`low_volume_position_only`, or `imp_90d` under a few hundred).
4. **Check the client is not mid-migration or in a portfolio-wide regime shift** — if the whole account moved, the page-level signal is drowned out (see the §4 alarm).
5. **Decide the action themselves** — refresh / expand / rewrite metadata / leave. The queue supplies the order and the reasons, not the edit.

### The no-go list

1. **Never auto-edit or auto-publish.** Every row keeps a human accept / reject. *(ML-09: "the evidence supports testing an ordering of human attention; it does not support automating actions.")*
2. **Never refresh a `durable_riser_hold`.** Its traffic is *rising* — route to monitor.
3. **Never act on `low_volume_position_only` on the position signal alone.**
4. **Never present the score as "this page will decline."** It is an ordering over a population.
5. **Never claim a refresh prevents a decline.** No experiment was run.
6. **Never rank across clients without per-client context.** A global top-50 can be one client's bad month (ML-07).
7. **Never run the queue while the §4 base-rate alarm is tripped** without a human re-review and refit.
8. **Never call `final_score` a probability.** Calibration was not evaluated.

The cell below quantifies the guards on the top 50.

In [7]:
top50 = e.head(50)
held = top50['_codes'].apply(lambda c: ('durable_riser_hold' in c) or ('low_volume_position_only' in c))
print(f'top-50 queue:')
print(f"  durable_riser_hold        : {int(top50['_codes'].apply(lambda c: 'durable_riser_hold' in c).sum())}")
print(f"  low_volume_position_only  : {int(top50['_codes'].apply(lambda c: 'low_volume_position_only' in c).sum())}")
print(f"  -> routed to hold / watchlist by the no-go rules: {int(held.sum())} of 50")
print(f"  confidence = high         : {int((top50['confidence'] == 'high').sum())}")
print(f"\ntop-20 OOF false positives (ranked high, did NOT decline): "
      f"{int((top20['label_decline'] == 0).sum())} of 20  (hit rate {top20['label_decline'].mean():.2f} vs base {BASE:.2f})")
fp = top50[top50['label_decline'] == 0].head(3)
print('\nexample high-rank misses (what a human review catches):')
print(fp[['final_rank', 'content_hash_id', 'reason_codes', 'imp_90d', 'last30_vs_pace', 'pos_gap', 'days_since_update']].to_string(index=False))

top-50 queue:
  durable_riser_hold        : 13
  low_volume_position_only  : 8
  -> routed to hold / watchlist by the no-go rules: 15 of 50
  confidence = high         : 5

top-20 OOF false positives (ranked high, did NOT decline): 15 of 20  (hit rate 0.25 vs base 0.04)

example high-rank misses (what a human review catches):
 final_rank          content_hash_id                      reason_codes  imp_90d  last30_vs_pace  pos_gap  days_since_update
          1 content_62ee86ec78b7352d position_slipping|model_high_risk    429.0            1.20 3.107643                  4
          5 content_a4fe6a8304641bd7                   model_high_risk    310.0            1.25 0.969549               <NA>
          7 content_01134b24adf8eea7 position_slipping|model_high_risk    485.0            1.01 4.399097                  4


## 4. Monitoring / retrain triggers

What would tell you the recommendations went stale — and what a tripped trigger does: **pause the next weekly queue pending a human re-review and a refit.** It never rescinds actions a person already took.

### Computed here

- **Outcome-drift alarm** — recompute the label base rate each month vs the deployed model's training rate (0.045); trip if it leaves `0.045 ± 0.05` **or** moves > 0.05 month-over-month. **It trips in 3 of the 4 months** (Apr 0.219, May 0.198, Jun 0.345 — Jun cited from `ml09_validation_audit.json`, not re-queried). The label is a panel-trend statistic; a monthly re-review is not optional.
- **PSI feature-drift harness** — Population Stability Index on the model's top features between the March training frame and April / May:

  | feature | PSI Mar→Apr | PSI Mar→May |
  |---|---|---|
  | `content_age_days` (RF's #1 feature) | 0.23 (major) | 0.44 (major) |
  | `pos_last30` | 0.16 (moderate) | 0.31 (major) |
  | `softening_ratio` | 0.06 | 0.11 (moderate) |

  The model's most important feature drifts **hard** month-over-month — which is a large part of why the model does not hold its edge over time.

### Proposed policy (not implemented here)

- Monthly refit on a rolling window — given the base-rate and feature drift above, this is the minimum viable operating procedure, not a nice-to-have.
- Per-client precision@10 tracked monthly; investigate any client below the base rate for two consecutive months.
- A deployment-time join to exclude pages a person optimised in the last N days, so the queue does not re-flag work in progress.

In [8]:
# --- feature frames for later months (June is NOT assembled -- it is the sealed month) ---
e_apr, X_apr, y_apr, g_apr = assemble('2026-04-01')
e_may, X_may, y_may, g_may = assemble('2026-05-01')

# June base rate: cited from the committed ML-09 receipt, not re-queried
ML09 = None
for cand in [Path('work/outputs/ml09_validation_audit.json'), Path('../outputs/ml09_validation_audit.json'),
             Path('outputs/ml09_validation_audit.json')]:
    if cand.is_file():
        ML09 = json.loads(cand.read_text()); break
jun_base = float(ML09['label_non_stationary']['base_rate_by_month']['2026-06']) if ML09 else float('nan')

TRAIN_BASE = 0.045
drift = pd.DataFrame({
    'month': ['2026-03', '2026-04', '2026-05', '2026-06 (sealed, cited)'],
    'base_rate': [round(BASE, 3), round(float(y_apr.mean()), 3), round(float(y_may.mean()), 3), round(jun_base, 3)],
})
drift['abs_dev_from_train'] = (drift['base_rate'] - TRAIN_BASE).abs().round(3)
drift['mom_change'] = drift['base_rate'].diff().abs().round(3)
drift['ALARM'] = (drift['abs_dev_from_train'] > 0.05) | (drift['mom_change'] > 0.05)
print('OUTCOME-DRIFT ALARM (train base rate = 0.045, band +/- 0.05):\n')
print(drift.to_string(index=False))
print(f"\n-> the alarm trips in {int(drift['ALARM'].sum())} of 4 months. The label is a panel-trend statistic;")
print("   a monthly re-review is a hard prerequisite for any deployment.")

def psi(expected, actual, bins=10):
    ex = pd.Series(expected).replace([np.inf, -np.inf], np.nan).dropna()
    ac = pd.Series(actual).replace([np.inf, -np.inf], np.nan).dropna()
    edges = np.unique(np.quantile(ex, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3: return np.nan
    e_pct = np.clip(np.histogram(ex, edges)[0] / len(ex), 1e-4, None)
    a_pct = np.clip(np.histogram(ac, edges)[0] / len(ac), 1e-4, None)
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))

PSI_FEATURES = ['content_age_days', 'softening_ratio', 'pos_last30']
psi_rows = [{'feature': f,
             'PSI Mar->Apr': round(psi(e[f], e_apr[f]), 3),
             'PSI Mar->May': round(psi(e[f], e_may[f]), 3)} for f in PSI_FEATURES]
psi_df = pd.DataFrame(psi_rows)
print('\nPSI FEATURE-DRIFT HARNESS (>0.10 moderate, >0.25 major):\n')
print(psi_df.to_string(index=False))

assemble(2026-04-01):  65,045 rows |  27 clients | base rate 0.2191


assemble(2026-05-01):  46,247 rows |  27 clients | base rate 0.1985
OUTCOME-DRIFT ALARM (train base rate = 0.045, band +/- 0.05):

                  month  base_rate  abs_dev_from_train  mom_change  ALARM
                2026-03      0.045               0.000         NaN  False
                2026-04      0.219               0.174       0.174   True
                2026-05      0.198               0.153       0.021   True
2026-06 (sealed, cited)      0.345               0.300       0.147   True

-> the alarm trips in 3 of 4 months. The label is a panel-trend statistic;
   a monthly re-review is a hard prerequisite for any deployment.

PSI FEATURE-DRIFT HARNESS (>0.10 moderate, >0.25 major):

         feature  PSI Mar->Apr  PSI Mar->May
content_age_days         0.234         0.437
 softening_ratio         0.055         0.111
      pos_last30         0.164         0.308


## 5. Exports for the paper

The ML-11 paper builds on these files:

| file | paper section it feeds |
|---|---|
| `work/outputs/ml10_action_queue.csv` (gitignored) | §7 Recommendation — the ranked queue |
| `work/outputs/ml10_playbook.json` (committed) | §7 Recommendation — taxonomy, tiers, no-go list, monitoring |
| `work/figures/fig1_precision_at_k.png` (committed) | Results — queue precision vs depth |
| `work/figures/fig2_base_rate_drift.png` (committed) | Limitations — the non-stationary label base rate |

In [9]:
OUT = next((c for c in [Path('work/outputs'), Path('../outputs'), Path('outputs')] if c.parent.exists()), Path('work/outputs'))
FIG = next((c for c in [Path('work/figures'), Path('../figures'), Path('figures')] if c.parent.exists()), Path('work/figures'))
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

# --- queue CSV (gitignored working artifact) ---
qcols = ['final_rank', 'content_hash_id', 'client_hash_id', 'final_score', 'rf_oof', 'rule_pct',
         'confidence', 'suggested_action', 'reason_codes', 'label_decline',
         'imp_90d', 'imp_last30', 'pace_30d', 'pos_last30', 'pos_gap', 'days_since_update', 'content_type']
e[qcols].rename(columns={'rf_oof': 'model_oof_score', 'rule_pct': 'rule_percentile'}) \
        .to_csv(OUT / 'ml10_action_queue.csv', index=False)

# --- Figure 1: precision@K (blend vs rule vs random) ---
Ks = list(range(10, 201, 10))
yv = e['label_decline'].to_numpy()
rand_s = np.random.default_rng(SEED).random(len(e))
series = {'blend (0.7 model + 0.3 rule)': e['final_score'].to_numpy(),
          'frozen rule only': e['rule_pct'].to_numpy(),
          'random': rand_s}
fig, ax = plt.subplots(figsize=(7, 4))
for name, s in series.items():
    ax.plot(Ks, [precision_at_k(s, yv, k) for k in Ks], marker='o', ms=3, label=name)
ax.axhline(BASE, ls='--', c='grey', lw=1, label=f'base rate {BASE:.3f}')
ax.set_xlabel('K (queue depth)'); ax.set_ylabel('precision@K  (grouped out-of-fold)')
ax.set_title('Queue precision vs depth (D = 2026-03-01), grouped out-of-fold')
ax.legend(fontsize=8); ax.grid(alpha=.3)
fig.savefig(FIG / 'fig1_precision_at_k.png', dpi=110, bbox_inches='tight'); plt.close(fig)

# --- Figure 2: label base rate drift ---
months = ['2026-03', '2026-04', '2026-05', '2026-06']
brs = [BASE, float(y_apr.mean()), float(y_may.mean()), jun_base]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(months, brs, marker='o', color='#c0392b')
ax.axhspan(TRAIN_BASE - 0.05, TRAIN_BASE + 0.05, alpha=.15, color='green',
           label='training base rate 0.045 ± 0.05')
for m, b in zip(months, brs):
    ax.annotate(f'{b:.3f}', (m, b), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
ax.set_ylabel('label base rate  (30-day sustained impressions decline)')
ax.set_title('The label base rate is non-stationary — the drift alarm trips every month')
ax.set_ylim(0, 0.42); ax.legend(fontsize=8); ax.grid(alpha=.3)
fig.savefig(FIG / 'fig2_base_rate_drift.png', dpi=110, bbox_inches='tight'); plt.close(fig)

# --- committed receipt ---
playbook = {
    'decision_date': '2026-03-01', 'seed': SEED, 'queue_rows': int(len(e)),
    'base_rate': round(BASE, 4),
    'library_versions': {'scikit-learn': sklearn.__version__, 'numpy': np.__version__,
                         'pandas': pd.__version__, 'matplotlib': matplotlib.__version__},
    'score_formula': '100 * (0.70 * model_oof_score + 0.30 * frozen_rule_percentile); model = ML-08 random forest, grouped OOF; '
                     'frozen rule = 0.40*traffic_softening + 0.30*position_slip + 0.30*reach_thinning (ML-07). NOT a calibrated probability.',
    'reason_code_taxonomy': {
        'high_exposure': 'imp_90d >= 500',
        'traffic_softening': 'imp_last30 < 0.8 * (imp_90d/3)',
        'position_slipping': 'avg position worsened >= 1.0, last 30d vs prior 60d',
        'reach_thinning': 'days_with_impressions fell >= 25%',
        'stale_content': 'days_since_update >= 180 (partial coverage)',
        'model_high_risk': 'model OOF score in the top 10%',
        'model_and_rule_agree': 'model_high_risk AND >= 2 rule codes',
        'durable_riser_hold': 'imp_last30 > 1.5 * pace -> monitor, never refresh',
        'low_volume_position_only': 'imp_90d < 250 AND only rule signal is position_slipping',
    },
    'suggested_action_map': 'durable_riser_hold->monitor_only; low_volume_position_only->watchlist; '
                            'traffic_softening|reach_thinning->refresh; position_slipping+high_exposure->review_position; '
                            'stale_content only->review_metadata; else->monitor',
    'confidence_tier_rules': 'high: final_score>=P80 AND imp_90d>=500 AND model_oof>=0.5 AND >=2 rule codes AND no policy hold; '
                             'medium: final_score>=P50; else low',
    'no_go_list': [
        'never auto-edit or auto-publish; every row keeps a human accept/reject',
        'never refresh a durable_riser_hold', 'never act on low_volume_position_only on position alone',
        'never present the score as "this page will decline"', 'never claim a refresh prevents decline',
        'never rank across clients without per-client context',
        'never run the queue while the base-rate alarm is tripped without re-review',
        'never call final_score a probability (calibration not evaluated)',
    ],
    'monitoring': {
        'outcome_drift_alarm': json.loads(drift.to_json(orient='records')),
        'psi_feature_drift': psi_df.to_dict(orient='records'),
        'proposed_not_implemented': ['monthly refit', 'per-client precision@10 tracking',
                                     'recently-optimised-page exclusion join at deployment'],
    },
    'queue_summary': {
        'action_mix': e['suggested_action'].value_counts().to_dict(),
        'confidence_mix': e['confidence'].value_counts().to_dict(),
        'top20_oof_hit_rate': round(float(top20['label_decline'].mean()), 3),
        'top50_held_for_human_judgement': int(held.sum()),
    },
    'figures': ['work/figures/fig1_precision_at_k.png', 'work/figures/fig2_base_rate_drift.png'],
    'population_note': 'churn-censored (clients absent from a label month excluded); GSC-visible pages with imp_last30 >= 100 only; '
                       'one content operation; ~17-month snapshot; no causal claim.',
}
(OUT / 'ml10_playbook.json').write_text(json.dumps(playbook, indent=2, default=str))
print('wrote', (OUT / 'ml10_action_queue.csv').resolve(), '(gitignored)')
print('wrote', (FIG / 'fig1_precision_at_k.png').resolve())
print('wrote', (FIG / 'fig2_base_rate_drift.png').resolve())
print('wrote', (OUT / 'ml10_playbook.json').resolve())
print('\n' + json.dumps(playbook, indent=2, default=str))

wrote /Users/ashishpal/Documents/flyrank/flyrank_ml_intern/work/outputs/ml10_action_queue.csv (gitignored)
wrote /Users/ashishpal/Documents/flyrank/flyrank_ml_intern/work/figures/fig1_precision_at_k.png
wrote /Users/ashishpal/Documents/flyrank/flyrank_ml_intern/work/figures/fig2_base_rate_drift.png
wrote /Users/ashishpal/Documents/flyrank/flyrank_ml_intern/work/outputs/ml10_playbook.json

{
  "decision_date": "2026-03-01",
  "seed": 42,
  "queue_rows": 58583,
  "base_rate": 0.045,
  "library_versions": {
    "scikit-learn": "1.9.0",
    "numpy": "2.5.1",
    "pandas": "3.0.5",
    "matplotlib": "3.11.1"
  },
  "score_formula": "100 * (0.70 * model_oof_score + 0.30 * frozen_rule_percentile); model = ML-08 random forest, grouped OOF; frozen rule = 0.40*traffic_softening + 0.30*position_slip + 0.30*reach_thinning (ML-07). NOT a calibrated probability.",
  "reason_code_taxonomy": {
    "high_exposure": "imp_90d >= 500",
    "traffic_softening": "imp_last30 < 0.8 * (imp_90d/3)",
    "positi

## Self-check

- [x] Queue ranks by a model + rule **blend**; every row carries reason codes, a suggested action, and a confidence tier; the score is called "model score", never "probability"
- [x] Intended-use statement + explicit "where it stops being valid" (churn-censored, GSC-visible ≥ 100/30d, non-stationary base rate, no causal claim, wide per-client variance)
- [x] Human-review checklist (5 steps) + an 8-item no-go list; both backed by counts from the top 50
- [x] Monitoring: outcome-drift alarm (trips every month) + a computed PSI harness (Mar→Apr, Mar→May); proposed policies flagged as proposed
- [x] **June 2026 base rate cited from `work/outputs/ml09_validation_audit.json`** — `month=2026-06` is never assembled; base rate printed next to every metric
- [x] Exports written: `ml10_action_queue.csv` (gitignored), `fig1`/`fig2` PNGs, `ml10_playbook.json`; seeds fixed (42); library versions recorded
- [x] IDs shown are hashes; no client names / URLs / raw queries
- [ ] Commit `work/notebooks/w07_action_playbook.ipynb` (with outputs) + `work/outputs/ml10_playbook.json` + `work/figures/*.png` (the queue CSV stays gitignored)